<a href="https://colab.research.google.com/github/alexlopespereira/idp_mestrado/blob/main/Aulas/Aula6/Aula6_Exercicio_Solucoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 6 - Exercícios - Pandas e Fontes de dados
## SOLUÇÕES

In [ ]:
# Importe as bibliotecas pandas, numpy
import pandas as pd
import numpy as np
from numpy import nan

### 6.1 Carregue o arquivo do IDEB 2017 no formato de um DataFrame. Crie uma função para calcular a média do IDEB de 2017 de todos os municípios de um determinado Estado.
#### Remova os registros cujo valor na coluna REDE sejam Municipal, Estadual ou Federal. Deixe somente os registros cujos valores na coluna REDE sejam Pública.
#### Dica 1: Antes de calcular, certifique-se de interpretar os valores '-' como NA durante o carregamento dos dados.
#### Dica 2: Use a função loc para fazer o filtro de registros.
#### Dica 3: Use o operador & (and) para congregar dois critérios de filtro.

In [ ]:
# Carregue os dados do IDEB
path_ideb='https://github.com/alexlopespereira/enapespcd2021/raw/main/data/originais/ideb/ideb_municipios2017.xlsx'
df_ideb = pd.read_excel(path_ideb, skiprows=9, skipfooter=3, na_values='-')

In [ ]:
def media_ideb(df, sigla_estado):
    """retorne o valor da media do IDEB de 2017 do Estado especificado por sigla_estado.
    """
    df.replace({'-': np.nan}, inplace=True)
    bool_estado = df['SG_UF'] == sigla_estado
    bool_publica = df['REDE'] == 'Pública'
    return df.loc[bool_estado & bool_publica, 'IDEB12_17'].mean()

In [ ]:
## Faça seus testes aqui
print(f"DF: {round(media_ideb(df_ideb, 'DF'), 1)}")  # 3.4
print(f"SP: {round(media_ideb(df_ideb, 'SP'), 1)}")  # 4.0
print(f"GO: {round(media_ideb(df_ideb, 'GO'), 1)}")  # 4.2

In [ ]:
# Validação
df_ideb = pd.read_excel(path_ideb, skiprows=9, skipfooter=3, na_values='-')
assert round(media_ideb(df_ideb, 'DF'), 1) == 3.4
df_ideb = pd.read_excel(path_ideb, skiprows=9, skipfooter=3, na_values='-')
assert round(media_ideb(df_ideb, 'SP'), 1) == 4.0
df_ideb = pd.read_excel(path_ideb, skiprows=9, skipfooter=3, na_values='-')
assert round(media_ideb(df_ideb, 'GO'), 1) == 4.2
print("Exercício 6.1 OK!")

### 6.2 Escreva uma função para gerar uma permutação (aleatória) de uma lista e colocar o resultado num dataframe com os elementos da lista agrupados em grupos de tamanho N.
#### A coluna de índices deve conter o nome dos grupos: Grupo 0, Grupo 1, Grupo 2, ...
#### Dica 1: A função len calcula o tamanho de uma lista.
#### Dica 2: Use a função math.ceil(N/n).

In [ ]:
def create_groups(names_list, n):
    """Crie um dataframe com os nomes da lista names_list agrupados em grupos de tamanho n.
    """
    from random import shuffle
    import math
    N = len(names_list)
    groups = ['Grupo {0}'.format(g) for g in range(math.ceil(N/n))] * n
    shuffle(names_list)
    groups.sort()
    return pd.DataFrame(index=groups[:N], data=names_list)

In [ ]:
## Faça seus testes aqui
# !pip install names  # Execute esta linha na primeira vez
import names
N = 9
group_length = 4
test_data = [names.get_full_name() for n in range(N)]
print(create_groups(test_data, group_length))

In [ ]:
# Validação
import names
N = 9
group_length = 4
test_data = [names.get_full_name() for n in range(N)]
result = create_groups(test_data, group_length)
expected_groups = ['Grupo 0', 'Grupo 0', 'Grupo 0', 'Grupo 0', 'Grupo 1', 'Grupo 1', 'Grupo 1', 'Grupo 1', 'Grupo 2']
assert result.index.to_list() == expected_groups
print("Exercício 6.2 OK!")

### 6.3 Escreva uma função para carregar corretamente o dataframe de população disponibilizado. Crie uma coluna chamada cod_ibge7 a partir da concatenação do conteúdo das colunas cod_uf e cod_munic.
#### Dica 1: Converta o codigo do município para string ao carregar o dataframe de população utilizando dtype={'cod_munic': str, 'cod_uf': str}.
#### Dica 2: Concatene o codigo da UF com o código do município usando o operador +.

In [ ]:
def load_pop(path_pop):
    """retorne um dataframe da população
    """
    df_pop = pd.read_excel(path_pop, sheet_name="Municipios", dtype={'cod_munic': str, 'cod_uf': str})
    df_pop['cod_ibge7'] = df_pop['cod_uf'] + df_pop['cod_munic']
    return df_pop

In [ ]:
## Faça seus testes aqui
path_pop = 'https://github.com/alexlopespereira/enapespcd2021/raw/main/data/originais/populacao/estimativa_dou_2017.xlsx'
print(load_pop(path_pop).head())

In [ ]:
# Validação
path_pop = 'https://github.com/alexlopespereira/enapespcd2021/raw/main/data/originais/populacao/estimativa_dou_2017.xlsx'
result = load_pop(path_pop).iloc[0]
assert result['uf'] == 'RO'
assert result['cod_uf'] == '11'
assert result['cod_munic'] == '00015'
assert result['cod_ibge7'] == '1100015'
print("Exercício 6.3 OK!")

### 6.4 Despivote o dataframe abaixo transformando as colunas 1991, 2000 e 2010 em conteúdos das linhas no dataframe resultante.

In [ ]:
def unpivot_gini(df):
    """retorne um dataframe despivotado.
    """
    return df.melt(id_vars=['Município'], var_name='data', value_name='gini')

In [ ]:
## Faça seus testes aqui
dataset_gini = {'Município': {0: "110001 Alta Floresta D'Oeste",
  1: '110037 Alto Alegre dos Parecis',
  2: '110040 Alto Paraíso',
  3: "110034 Alvorada D'Oeste",
  4: '110002 Ariquemes'},
 '1991': {0: 0.5983, 1: None, 2: None, 3: 0.569, 4: 0.5827},
 '2000': {0: 0.5868, 1: 0.508, 2: 0.6256, 3: 0.6534, 4: 0.5927},
 '2010': {0: 0.5893, 1: 0.5491, 2: 0.5417, 3: 0.5355, 4: 0.5496}}
df_gini = pd.DataFrame(dataset_gini)
print(unpivot_gini(df_gini))

In [ ]:
# Validação
dataset_gini = {'Município': {0: "110001 Alta Floresta D'Oeste",
  1: '110037 Alto Alegre dos Parecis',
  2: '110040 Alto Paraíso',
  3: "110034 Alvorada D'Oeste",
  4: '110002 Ariquemes'},
 '1991': {0: 0.5983, 1: None, 2: None, 3: 0.569, 4: 0.5827},
 '2000': {0: 0.5868, 1: 0.508, 2: 0.6256, 3: 0.6534, 4: 0.5927},
 '2010': {0: 0.5893, 1: 0.5491, 2: 0.5417, 3: 0.5355, 4: 0.5496}}
df_gini = pd.DataFrame(dataset_gini)
result = unpivot_gini(df_gini)
assert len(result) == 15  # 5 municípios x 3 anos
assert list(result.columns) == ['Município', 'data', 'gini']
print("Exercício 6.4 OK!")